# Part III — the forward problem: computational fluid dynamics with QTT

Companion notebook to the Part III slides (`slides/part3.pdf`) and to Section *Part III*
of the lecture notes. It follows the slides in order:

1. **Why fluid dynamics** (Navier–Stokes, Reynolds number, turbulence, scales and rank) — slides 5–11
2. **CFD framework** (flow initialization, masks, Chorin time integrator, readout, complexity) — slides 12–20
3. **Numerical results** (what the full solver does) — slides 21–23
4. **Recent literature** — slides 24–26
5. **Toward turbulence** (statistical metrics, what compression does to small scales) — slides 27–38

Exercises are interleaved where the concept is introduced. Each exercise is followed
directly by a cell marked **Solution**: try it first, predict, then measure.

The solver used here, `qtade_cfd.Flow`, is a teaching demonstrator: Chorin's projection
with every field a quantics tensor train and every operator an MPO. Keep `n = 6`
(a $64\times64$ grid) unless a cell says otherwise. The whole notebook runs in a few
minutes on a laptop.

In [ ]:
import os

# The tensors here are tiny; multithreaded BLAS only adds overhead (and on a shared
# machine it can make every step ten times slower). Must be set before numpy loads.
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import time

import matplotlib.pyplot as plt
import numpy as np

import qtade_cfd as cfd
import qtade_quimb as qq
import qtade_tn as tn

plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.grid": True, "grid.alpha": 0.3,
                     "font.size": 10})

## 1. Why fluid dynamics

After Parts I and II we have every QTT primitive needed to solve a PDE. *Which* PDE is
not a trivial question, because QTT compresses information in a peculiar way:

* in an MPS of quantum degrees of freedom, rank = non-factorizability = entanglement,
  and the area law makes ground states cheap;
* in a QTT the sites are the binary digits of the grid index, i.e. **length scales**,
  so rank measures **inter-scale correlations**.

So the right question is: which physics has weak correlations between scales? That
leads to turbulence — the computational bottleneck of CFD.

### The Navier–Stokes equations

$$\frac{\partial\mathbf{v}}{\partial t} + (\mathbf{v}\cdot\nabla)\,\mathbf{v}
  = -\frac{1}{\varrho}\nabla p + \nu\,\nabla^2\mathbf{v},
  \qquad \nabla\cdot\mathbf{v} = 0 .$$

Advection (nonlinear), pressure (with constant density $\varrho$: incompressible),
dissipation (viscosity $\nu$; $\nu = 0$ is the inviscid Euler limit), and the
divergence-free condition. One control parameter survives nondimensionalization,

$$\mathrm{Re} \sim \frac{\text{advection}}{\text{dissipation}} \sim \frac{UL}{\nu},$$

laminar at small $\mathrm{Re}$, turbulent for $\mathrm{Re}\gg 1$.

### Turbulence and the number of scales

Richardson's cascade and Kolmogorov (1941): energy injected at the large scale $L$ is
passed down, self-similarly through the inertial range ($E(k)\sim k^{-5/3}$), to the
dissipative scale $\eta$, with

$$L/\eta \sim \mathrm{Re}^{3/4}, \qquad
  \#\text{dof} \sim (L/\eta)^{D} = \mathrm{Re}^{3D/4}
  \;\;(= \mathrm{Re}^{9/4} \text{ in 3D}).$$

In quantics the number of binary **scales** per axis is $M = \log_2(L/\eta)$. Compute the
table on the slide.

In [ ]:
print(f"{'Re':>8} {'# scales M = log2(Re^3/4)':>27} {'3D DNS points Re^9/4':>22}")
for e in range(4, 10):
    Re = 10.0 ** e
    print(f"{'1e%d' % e:>8} {0.75 * np.log2(Re):>27.1f} {Re ** 2.25:>22.1e}")

A typical airfoil sits at $\mathrm{Re}\sim 2\times10^{6}$–$2\times10^{7}$, i.e.
$10^{13}$–$10^{16}$ grid points for DNS: impossible, hence RANS/LES models. But only
**about 20 scales**. A QTT stores $O(M\chi^2)$ numbers; if the correlations between scales
are mostly *local* (stronger coupling between neighbouring scales, self-similarity in the
inertial range), $\chi$ stays small and the memory is logarithmic in the mesh size. This
is the idea of Gourianov *et al.* (2022).

---
### Exercise 1 — rank measures inter-scale correlations

Build synthetic 2D "turbulent" fields on a $256\times256$ grid with a $k^{-5/3}$ energy
spectrum truncated at $k_{\max}$ (the generator is given). Then:

1. compress to relative tolerance $10^{-2}$ and plot the bond dimension at **every**
   bipartition of the (interleaved) chain, for $k_{\max} = 8$ and $k_{\max} = 64$
   (Gourianov's Fig. 1);
2. tabulate the maximal $\chi$ against $k_{\max} \in \{4, 8, 16, 32, 64\}$.

Predict first: does $\chi$ track the grid, or the width of the inertial range?

In [ ]:
m = 8
Nt = 2 ** m
kf = np.fft.fftfreq(Nt) * Nt
KX, KY = np.meshgrid(kf, kf, indexing="ij")
K = np.hypot(KX, KY)
K[0, 0] = 1.0


def synthetic_field(kmax, seed=7):
    # amplitude k^(-5/6) in 2D gives E(k) ~ k * k^(-5/3) / k ... close enough to K41 for us
    rng = np.random.default_rng(seed)
    phase = rng.standard_normal((Nt, Nt)) + 1j * rng.standard_normal((Nt, Nt))
    spec = K ** (-5 / 6)
    spec[K > kmax] = 0.0
    f = np.real(np.fft.ifft2(spec * phase))
    return f / f.std()


# TODO: qq.from_grid(field, eps=1e-2) and tn.tt_ranks(...) are all you need.

#### Solution — Exercise 1

In [ ]:
plt.figure(figsize=(6, 3))
for kmax in (8, 64):
    r = tn.tt_ranks(qq.from_grid(synthetic_field(kmax), eps=1e-2))
    plt.semilogy(range(1, len(r) - 1), r[1:-1], "o-", label=f"$k_{{max}}={kmax}$")
plt.semilogy(range(1, 2 * m), [min(2 ** k, 2 ** (2 * m - k)) for k in range(1, 2 * m)],
             "k--", lw=1, label="maximal rank")
plt.xlabel("bipartition (after site k)"), plt.ylabel(r"$\chi_k$"), plt.legend(fontsize=8)
plt.tight_layout()

print(f"{'k_max (inertial range)':>24} {'chi @ 1e-2':>12} {'chi @ 1e-3':>12}")
for kmax in (4, 8, 16, 32, 64):
    f = synthetic_field(kmax)
    c2 = max(tn.tt_ranks(qq.from_grid(f, eps=1e-2)))
    c3 = max(tn.tt_ranks(qq.from_grid(f, eps=1e-3)))
    print(f"{kmax:>24} {c2:>12} {c3:>12}")

The rank profile is a tent: small at the ends (few scales on one side), largest in the
middle where coarse and fine scales are cut apart. The maximum climbs with the **width of
the inertial range** (the thing $\mathrm{Re}$ controls), not with the grid (the thing
quantics already handles). Whether real turbulence keeps $\chi$ manageable is therefore
an empirical question — and it is the question of Section 5.

## 2. CFD framework

CFD pipelines are very versatile. Can we replicate one inside QTT, with better scaling?
We follow Peddinti *et al.* (2023/2024), a five-step loop:

1. **flow initialization** — encode $\mathbf{v}$ as an MPS;
2. **mask application** — approximate MPS masks of arbitrary objects;
3. **divergence-free projection** and 4. **time integration** — Chorin's method with open
   boundary conditions, iterated $T$ times;
5. **MPS solution oracle** — efficient retrieval without decompressing.

### 2.1 First step: flow initialization (QTT encoding recap)

$$v(x,y) \xrightarrow{\text{discretize}} v(\mathbf{i},\mathbf{j})
\xrightarrow{\text{amplitude encoding}} v_{\mathbf{i},\mathbf{j}}\,|\mathbf{i},\mathbf{j}\rangle,
\qquad
v_{\mathbf{i},\mathbf{j}} = A_1^{(i_1)}\cdots A_{n}^{(i_{n})}\,B_1^{(j_1)}\cdots B_{n}^{(j_{n})} .$$

The slides use the **2D stacked** order: all $x$ bits, then all $y$ bits. Bit $i_1$ picks
the left/right half of the domain, $i_2$ the quarter inside it, and so on: *qubits ⇔
scales, entanglement ⇔ inter-scale correlations*, and storage drops from $O(2^N)$ to
$O(N\chi^2)$. The alternative **interleaved** order $i_1 j_1 i_2 j_2\dots$ keeps the $x$ and
$y$ bits of the same scale next to each other; `qtade_cfd` uses it by default.

A single amplitude is one product of matrices, e.g. $v_{\mathbf{110},\mathbf{10}}$ on an
$8\times4$ grid is the chain selected by the bit string $|11010\rangle$:

In [ ]:
n = 6
N = 2 ** n
xx = (np.arange(N) + 0.5) / N
X, Y = np.meshgrid(xx, xx, indexing="ij")
f = np.exp(-30 * ((X - 0.4) ** 2 + (Y - 0.6) ** 2)) * np.sin(6 * np.pi * X * Y)

c_st = qq.from_grid(f, eps=1e-10, ordering="stacked")
c_il = qq.from_grid(f, eps=1e-10, ordering="interleaved")

i, j = 0b110010, 0b101101
bi = [int(b) for b in f"{i:0{n}b}"]
bj = [int(b) for b in f"{j:0{n}b}"]
print("stacked bits     :", "".join(map(str, bi + bj)))
print("interleaved bits :", "".join(str(b) for pair in zip(bi, bj) for b in pair))
print(f"dense v[i, j]    = {f[i, j]:+.10f}")
print(f"stacked MPS      = {qq.evaluate(c_st, bi + bj):+.10f}")
print(f"interleaved MPS  = {qq.evaluate(c_il, [b for p in zip(bi, bj) for b in p]):+.10f}")

---
### Exercise 2 — stacked or interleaved?

The bit order is a modelling choice, and it changes the rank. On a $256\times256$ grid,
compress each field below at tolerance $10^{-6}$ in both orders and compare $\chi$:

* an isolated Gaussian vortex $e^{-40[(x-0.4)^2 + (y-0.6)^2]}$,
* a horizontal shear layer $\tanh((y-0.5)/0.02)$,
* a **diagonal** shear layer $\tanh((x-y)/0.02)$,
* $\sin(6\pi xy)$.

Which order wins for which field, and why? (Hint: what is the stacked rank of a
separable function $g(x)h(y)$?)

In [ ]:
n8 = 8
x8 = (np.arange(2 ** n8) + 0.5) / 2 ** n8
X8, Y8 = np.meshgrid(x8, x8, indexing="ij")
fields = {
    "Gaussian vortex": np.exp(-40 * ((X8 - 0.4) ** 2 + (Y8 - 0.6) ** 2)),
    "horizontal shear": np.tanh((Y8 - 0.5) / 0.02),
    "diagonal shear": np.tanh((X8 - Y8) / 0.02),
    "sin(6 pi x y)": np.sin(6 * np.pi * X8 * Y8),
}
# TODO

#### Solution — Exercise 2

In [ ]:
print(f"{'field':>18} {'stacked':>8} {'interleaved':>12}")
for name, fld in fields.items():
    r = [max(tn.tt_ranks(qq.from_grid(fld, eps=1e-6, ordering=o)))
         for o in ("stacked", "interleaved")]
    print(f"{name:>18} {r[0]:>8} {r[1]:>12}")

A separable field $g(x)h(y)$ has rank 1 across the $x|y$ cut, so in stacked order its
rank is just that of $g$ and $h$ separately: stacked wins for axis-aligned structure.
A diagonal feature couples $x$ and $y$ *at every scale*; stacked order must carry all of
it across the middle bond, while interleaved order keeps each scale local. Turbulence has
no preferred axis, which is why the turbulence studies of Section 5 favour interleaving.

### 2.2 Masking arbitrary objects

An obstacle enters as a mask that multiplies the velocity: 0 inside the body, 1 in the
fluid. We never have the mask as a dense array; we have a function we can *evaluate*, so
we build the MPS with **TT-cross** (Oseledets & Tyrtyshnikov 2010; Part II), at cost
$O(N\chi^2)\,(\text{cost of one evaluation}) + O(N\chi^3)$.

The sharp indicator $\theta(x)$ is a poor target: a curved discontinuity couples every
scale. The smoothed mask

$$m(x) = 1 - e^{-\alpha\,(q + |q|)}, \qquad q < 0 \text{ inside the body},$$

is exactly zero inside and relaxes to one over a width $\sim 1/\alpha$ outside. TT-cross
it at fixed rank and compare both against $\theta$ (slide 17).

In [ ]:
n9 = 9
N9 = 2 ** n9
w9 = 2.0 ** -np.arange(1, n9 + 1)


def level(x, y):
    # Sign convention: q must be NEGATIVE inside the solid, so that q + |q| = 0 there
    # and the mask vanishes in the body. Flip it and you will faithfully simulate a
    # flow that exists only inside the cylinder.
    return (x - 0.35) ** 2 + (y - 0.5) ** 2 - 0.15 ** 2


def coords(bits):          # interleaved bits x1 y1 x2 y2 ... -> cell centres
    return bits[:, 0::2] @ w9 + 0.5 / N9, bits[:, 1::2] @ w9 + 0.5 / N9


theta = lambda b: (level(*coords(b)) > 0).astype(float)
smooth = lambda b, a=300.0: 1 - np.exp(-a * (lambda q: q + np.abs(q))(level(*coords(b))))

x9 = (np.arange(N9) + 0.5) / N9
TH = (level(*np.meshgrid(x9, x9, indexing="ij")) > 0).astype(float)
fig, ax = plt.subplots(1, 2, figsize=(8, 3.6))
for a, (name, fn) in zip(ax, [(r"$|\mathrm{MPS}_\theta - \theta|$", theta),
                              (r"$|\mathrm{MPS}_m - \theta|$", smooth)]):
    c, stats = tn.tt_cross(fn, [2] * (2 * n9), rank=48, sweeps=3, return_stats=True)
    err = np.abs(qq.to_grid(c) - TH)
    im = a.imshow(err.T, origin="lower", cmap="coolwarm", vmin=0, vmax=1)
    a.set_title(name), a.set_xticks([]), a.set_yticks([])
    print(f"{name:>28}: chi = {max(tn.tt_ranks(c))}, {stats['evaluations']:,d} evaluations "
          f"(grid has {N9 ** 2:,d}), mean error {err.mean():.4f}, max error {err.max():.2f}")
plt.colorbar(im, ax=ax, shrink=0.8)

At the same rank the sharp mask leaves blocky $O(1)$ errors scattered over the body
(TT-cross has no rank to spend on the curved jump and overshoots elsewhere), while the
smoothed mask is wrong only in a thin ring at the interface — where it was *designed* to
differ from $\theta$. Curvature, not sharpness, is what costs rank.

### 2.3 Time integrator: Chorin's projection inside the QTT representation

* finite-difference scheme; derivatives and Laplacian as low-rank MPOs;
* MPO encoding of the boundary conditions (open BCs in the full solver; homogeneous
  Dirichlet walls in our demonstrator);
* **Chorin's projection** — predict and correct:

$$\mathbf{v}^{*}_{t+\Delta t} = \mathbf{v}_t + \big(-(\mathbf{v}_t\cdot\nabla)\mathbf{v}_t + \nu\nabla^2\mathbf{v}_t\big)\,\Delta t,$$
$$\nabla^2 p_{t+\Delta t} = \frac{\varrho}{\Delta t}\,\nabla\cdot\mathbf{v}^{*}_{t+\Delta t},$$
$$\mathbf{v}_{t+\Delta t} = \mathbf{v}^{*}_{t+\Delta t} - \frac{\Delta t}{\varrho}\nabla p_{t+\Delta t};$$

* the pressure Poisson equation is a linear system, solved with **DMRG** (Part II):
  $O(N\chi^6)$ worst case, and the dominant cost.

Pressure is not dynamical: it is whatever enforces $\nabla\cdot\mathbf{v}=0$ instantly, so
every step contains one globally coupled elliptic solve. Everything else is local
arithmetic plus rounding. (`qtade_cfd` sets $\varrho = 1$.)

A detail that is easy to get wrong and expensive to debug: the pressure operator must be
$\mathrm{div}\circ\mathrm{grad}$ built from **the same** difference operators used for the
gradient and the divergence. Substitute the compact three-point Laplacian and the
projection stops projecting.

In [ ]:
flow = cfd.Flow(n, nu=2e-3, dt=0.4 / N, chi_max=32, cutoff=1e-7, poisson_sweeps=2)
print(f"grid                 : {N} x {N} = {N * N:,d} cells, {2 * n} binary sites")
print(f"d/dx, d/dy  MPO rank : {max(tn.mpo_ranks(flow.Dx))}")
print(f"Laplacian   MPO rank : {max(tn.mpo_ranks(flow.L))}")
print(f"div(grad)   MPO rank : {max(tn.mpo_ranks(flow.Lp))}")

# A vortex pair. np.gradient uses a different stencil from our MPOs, so the initial
# field is visibly non-solenoidal -- which makes the projection easy to see working.
xg = np.linspace(0, 1, N, endpoint=False)
Xg, Yg = np.meshgrid(xg, xg, indexing="ij")
psi = 0.05 * np.sin(2 * np.pi * Xg) * np.sin(2 * np.pi * Yg)


def initial_velocity():
    return (qq.from_grid(np.gradient(psi, xg, axis=1), eps=1e-9),
            qq.from_grid(-np.gradient(psi, xg, axis=0), eps=1e-9))


u, v = initial_velocity()
print(f"\nbefore one Chorin step: |div v| = {tn.tt_norm(cfd.divergence(flow, u, v)):.3e}")
u, v, p = flow.step(u, v)
print(f"after  one Chorin step: |div v| = {tn.tt_norm(cfd.divergence(flow, u, v)):.3e}")
print(f"pressure train: chi = {max(tn.tt_ranks(p))}")

Three orders of magnitude from one linear solve.

---
### Exercise 3 — what the nonlinearity actually costs

The advection term is the only place where ranks *multiply* rather than add. In
`Flow.step` it is `u * (Dx u) + v * (Dy u)`. For the initial field, record the bond
dimension of `u`, of `Dx u` after rounding, of the Hadamard product `u * (Dx u)` **before**
rounding (predict it first), and after rounding.

Then compare with the conservative form $\tfrac12\partial_x(u^2)$: identical in exact
arithmetic, but is it identical under truncation?

In [ ]:
flow = cfd.Flow(n, nu=2e-3, dt=0.4 / N, chi_max=32, cutoff=1e-7)
u, v = initial_velocity()
# TODO: tn.mpo_apply, tn.tt_hadamard and tn.tt_round are the pieces you need.

#### Solution — Exercise 3

In [ ]:
du = tn.tt_round(tn.mpo_apply(flow.Dx, u), eps=1e-7)
raw = tn.tt_hadamard(u, du)
rounded = tn.tt_round(raw, eps=1e-7)
print(f"chi(u)              = {max(tn.tt_ranks(u))}")
print(f"chi(Dx u)           = {max(tn.tt_ranks(du))}")
print(f"chi(u * Dx u) raw   = {max(tn.tt_ranks(raw))}   "
      f"(= {max(tn.tt_ranks(u))} x {max(tn.tt_ranks(du))}: ranks multiply)")
print(f"chi(u * Dx u) round = {max(tn.tt_ranks(rounded))}")

cons = tn.tt_scale(tn.tt_round(tn.mpo_apply(flow.Dx, tn.tt_round(tn.tt_hadamard(u, u),
                                                                 eps=1e-7)), eps=1e-7), 0.5)
diff = tn.tt_norm(tn.tt_add(rounded, tn.tt_scale(cons, -1))) / tn.tt_norm(rounded)
print(f"\nconservative form: chi(d/dx (u^2)/2) = {max(tn.tt_ranks(cons))}, "
      f"relative difference to advective form = {diff:.2e}")

Both forms cost one Hadamard and one MPO application. The conservative form rounds the
square before differentiating, the advective form rounds the product after, and on a
discrete grid $u\,D u \ne \tfrac12 D(u^2)$ anyway. Which is better depends on the field:
measure, do not assume.

#### Where the time goes

Run the demonstrator for a few steps and time the three stages.

In [ ]:
flow = cfd.Flow(n, nu=2e-3, dt=0.4 / N, chi_max=32, cutoff=1e-7, poisson_sweeps=1)
u, v = initial_velocity()
n_steps = 15
div_hist, frames = [], {}
t0 = time.perf_counter()
for k in range(n_steps):
    u, v, p = flow.step(u, v)
    div_hist.append(tn.tt_norm(cfd.divergence(flow, u, v)))
    if k in (0, n_steps // 2, n_steps - 1):
        frames[k] = qq.to_grid(u)
wall = time.perf_counter() - t0
print(f"{n_steps} steps in {wall:.1f} s ({wall / n_steps:.2f} s/step)")
for stage, t in flow.timings.items():
    print(f"  {stage:>11}: {t:6.2f} s ({100 * t / sum(flow.timings.values()):5.1f}%)")
print(f"\nPoisson solve: {100 * flow.poisson_share():.0f}% of the runtime "
      "(Peddinti et al. report 80.1% for the full solver).")

fig, ax = plt.subplots(1, 4, figsize=(12, 3))
for a, (k, fr) in zip(ax, frames.items()):
    a.imshow(fr.T, origin="lower", cmap="RdBu_r")
    a.set_title(f"$u$ at step {k}"), a.set_xticks([]), a.set_yticks([])
ax[3].semilogy(div_hist)
ax[3].set(xlabel="step", ylabel=r"$\|\nabla\cdot\mathbf{v}\|$", title="the constraint")
plt.tight_layout()

Any effort spent anywhere other than the elliptic solve is, to first order, wasted —
which is why preconditioning in TT format is the open problem with the largest payoff.

---
### Exercise 4 — truncation does not respect the constraint

SVD truncation is optimal in the Frobenius norm, and the Frobenius norm knows nothing
about $\nabla\cdot\mathbf{v}=0$. Run 10 steps at `chi_max` $\in\{8, 16, 32\}$ and plot
$\|\nabla\cdot\mathbf{v}\|$ against step for each. Which `chi_max` keeps the divergence
below $10^{-1}$? What does that cost in memory relative to the dense $64\times64$ field?

In [ ]:
# TODO: cfd.Flow(..., chi_max=chi, cutoff=1e-9), flow.step, cfd.divergence, tn.tt_size

#### Solution — Exercise 4

In [ ]:
plt.figure(figsize=(7, 3.2))
for chi in (8, 16, 32):
    fl = cfd.Flow(n, nu=2e-3, dt=0.4 / N, chi_max=chi, cutoff=1e-9, poisson_sweeps=1)
    uu, vv = initial_velocity()
    hist = []
    for _ in range(10):
        uu, vv, _ = fl.step(uu, vv)
        hist.append(tn.tt_norm(cfd.divergence(fl, uu, vv)))
    plt.semilogy(hist, label=f"$\\chi_{{max}}={chi}$")
    print(f"chi_max={chi:>3}: final |div v| = {hist[-1]:.2e}, "
          f"params(u) = {tn.tt_size(uu):>7,d} vs dense {N ** 2:,d}")
plt.xlabel("step"), plt.ylabel(r"$\|\nabla\cdot\mathbf{v}\|$"), plt.legend(fontsize=8)
plt.tight_layout()

Every rounding step puts a little divergence back and the next projection removes it;
`chi_max` sets the balance. And note the last column: on a $64\times64$ grid a train with
$\chi = 32$ stores *more* numbers than the dense field. Compression is an
*asymptotic* statement — $n\chi^2$ beats $4^n$ only once $n$ is large. Structure-preserving
truncation (rounding that keeps the constraint) is open.

---
### Exercise 5 — put an obstacle in the flow

Geometry costs one extra Hadamard product per step. Build the smoothed disc mask of
Section 2.2 on the $64\times64$ grid with TT-cross, apply it to both velocity components
after every step (10 steps), and check

1. that the velocity really is (nearly) zero inside the disc,
2. how much the mask raises the bond dimension of the flow.

In [ ]:
w6 = 2.0 ** -np.arange(1, n + 1)
inside = level(X, Y) < 0          # X, Y: cell centres of the 64 x 64 grid (Section 2.1)


def mask_fn(bits, alpha=300.0):
    q = level(bits[:, 0::2] @ w6 + 0.5 / N, bits[:, 1::2] @ w6 + 0.5 / N)
    return 1 - np.exp(-alpha * (q + np.abs(q)))


# TODO: mask = tn.tt_cross(mask_fn, [2] * (2 * n), rank=...), then in the loop
#       u = tn.tt_round(tn.tt_hadamard(mask, u), eps=..., chi_max=...)

#### Solution — Exercise 5

In [ ]:
mask = tn.tt_round(tn.tt_cross(mask_fn, [2] * (2 * n), rank=24, sweeps=3), eps=1e-8)
fl = cfd.Flow(n, nu=2e-3, dt=0.4 / N, chi_max=32, cutoff=1e-8, poisson_sweeps=1)
uu, vv = initial_velocity()
for _ in range(10):
    uu, vv, _ = fl.step(uu, vv)
    uu = tn.tt_round(tn.tt_hadamard(mask, uu), eps=1e-8, chi_max=32)
    vv = tn.tt_round(tn.tt_hadamard(mask, vv), eps=1e-8, chi_max=32)
grid = qq.to_grid(uu)
print(f"chi(mask) = {max(tn.tt_ranks(mask))}, chi(u) = {max(tn.tt_ranks(uu))} "
      f"(rank history {fl.rank_history})")
print(f"max |u| inside the disc = {np.abs(grid[inside]).max():.2e}, "
      f"mean |u| outside = {np.abs(grid[~inside]).mean():.2e}")

plt.figure(figsize=(4, 3.5))
plt.imshow(grid.T, origin="lower", cmap="RdBu_r")
plt.contour(inside.T.astype(float), levels=[0.5], colors="k", linewidths=1)
plt.xticks([]), plt.yticks([]), plt.title("$u$ with a masked obstacle")
plt.tight_layout()

The mask itself is cheap; what it costs is that the flow now has a curved boundary layer,
and the velocity rank saturates the cap. The full solver of Peddinti *et al.* applies the
mask exactly this way, plus ghost-cell inlet/outlet conditions inside the MPOs.

### 2.4 Recover the solution: the MPS solution oracle

If you have to rebuild the $2^{2n}$ array to look at the answer, the compression bought
nothing. Two readouts never form it:

* **coarse-grained evaluation** — contract the finest bits against $(1/2, 1/2)$, which is
  exactly average pooling: $O(N\chi^3)$;
* **pixel sampling** — draw grid points $i_1 i_2\dots \sim P(\mathbf{i}) \propto |v_\mathbf{i}|^2$,
  bit by bit from conditional marginals, as one samples a quantum state:
  $O(N_{\text{samples}}\,N\chi^3)$ (slide: $O(\chi^3)$ per site and sample).

A single point costs one matrix chain, $O(N\chi^2)$.

In [ ]:
print(f"{'n':>3} {'cells':>12} {'chi':>5} {'1 point (us)':>13} {'16x16 field (ms)':>17} "
      f"{'full field (ms)':>16}")
for ni in (5, 6, 7, 8, 9, 10):
    Ni = 2 ** ni
    xi = (np.arange(Ni) + 0.5) / Ni
    Xi, Yi = np.meshgrid(xi, xi, indexing="ij")
    ci = qq.from_grid(np.exp(-30 * ((Xi - 0.4) ** 2 + (Yi - 0.6) ** 2))
                      * np.sin(6 * np.pi * Xi * Yi), eps=1e-8)
    bits = [1, 0] * ni
    t0 = time.perf_counter()
    for _ in range(1000):
        qq.evaluate(ci, bits)
    t_point = (time.perf_counter() - t0) / 1000 * 1e6
    t0 = time.perf_counter()
    for _ in range(5):
        qq.to_grid(qq.coarse_grain(qq.to_mps(ci), 4))
    t_coarse = (time.perf_counter() - t0) / 5 * 1e3
    t0 = time.perf_counter()
    for _ in range(5):
        qq.to_grid(ci)
    t_full = (time.perf_counter() - t0) / 5 * 1e3
    print(f"{ni:>3} {Ni ** 2:>12,d} {max(tn.tt_ranks(ci)):>5} {t_point:>13.1f} "
          f"{t_coarse:>17.2f} {t_full:>16.2f}")

The first two timings barely move while the grid grows a thousandfold; the last tracks
the cell count. Build the diagnostics you need — averages, probes, samples, spectra — to
work on the train directly.

---
### Exercise 6 — pixel sampling

Write `sample_pixels(cores, n_samples)` that draws interleaved bit strings with
probability $|v_\mathbf{i}|^2/\|v\|^2$ without forming the dense field:

1. precompute right environments $R_k = \sum_{i_k\dots i_N} A^{(i_k)}\cdots A^{(i_N)}
   (A^{(i_k)}\cdots A^{(i_N)})^{T}$;
2. walk left to right: for each candidate bit $b$ the unnormalised marginal is
   $\ell\,A_k^{(b)} R_{k+1} (\ell\,A_k^{(b)})^{T}$, where $\ell$ is the (normalised)
   product of the cores already fixed; draw $b$, update $\ell$.

Apply it to the **vorticity** $\omega = \partial_x v - \partial_y u$ of the masked flow from
Exercise 5 (computed as a train), and check the sample histogram against $|\omega|^2$ on a
coarse $8\times8$ grid.

In [ ]:
omega = tn.tt_round(tn.tt_add(tn.mpo_apply(fl.Dx, vv),
                              tn.tt_scale(tn.mpo_apply(fl.Dy, uu), -1.0)), eps=1e-8)


def sample_pixels(cores, n_samples, seed=0):
    # TODO
    raise NotImplementedError

#### Solution — Exercise 6

In [ ]:
def sample_pixels(cores, n_samples, seed=0):
    rng = np.random.default_rng(seed)
    R = [np.ones((1, 1))]                                   # R[k]: environment of sites k..
    for c in reversed(cores):
        R.insert(0, np.einsum("aib,bc,dic->ad", c, R[0], c))
    left = np.ones((n_samples, 1))
    bits = np.zeros((n_samples, len(cores)), dtype=int)
    for k, c in enumerate(cores):
        cand = np.einsum("sa,aib->sib", left, c)            # (samples, bit, bond)
        p = np.einsum("sib,bc,sic->si", cand, R[k + 1], cand)
        b = (rng.random(n_samples) < p[:, 1] / p.sum(axis=1)).astype(int)
        bits[:, k] = b
        left = cand[np.arange(n_samples), b]
        left /= np.linalg.norm(left, axis=1, keepdims=True)  # avoid under/overflow
    return bits


t0 = time.perf_counter()
bits = sample_pixels(omega, 20000)
print(f"20,000 samples in {time.perf_counter() - t0:.2f} s")
weights = 2 ** np.arange(n - 1, -1, -1)
ix, iy = bits[:, 0::2] @ weights, bits[:, 1::2] @ weights

W = qq.to_grid(omega)                       # only to check the sampler
P = (W ** 2).reshape(8, N // 8, 8, N // 8).sum(axis=(1, 3))
H = np.histogram2d(ix, iy, bins=8, range=[[0, N], [0, N]])[0]
print(f"max |empirical - exact| on the 8x8 grid: {np.abs(H / H.sum() - P / P.sum()).max():.4f}")

fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
lim = np.abs(W).max()
ax[0].imshow(W.T, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
ax[0].set_title(r"$\omega$ (full resolution)")
ax[1].scatter(ix[:2000], iy[:2000], s=1, c="crimson")
ax[1].set(xlim=(0, N), ylim=(0, N), aspect="equal", title="2,000 sampled pixels")
for a in ax:
    a.set_xticks([]), a.set_yticks([])
plt.tight_layout()

The samples concentrate where the vorticity lives — the boundary layer of the obstacle
and the vortex cores — exactly as in the slide's pixel-sampling panel.

### 2.5 Complexity of the algorithms

From Peddinti *et al.* (2024):

| algorithmic task | complexity | runtime % |
|---|---|---|
| mask generation | $O(N\chi^3)$ | offline |
| mask application | $O(\chi^6)$ | 2.0 |
| divergence-free projection | $O(N\chi^6)$ | **80.1** |
| Euler time stepping | $O(\chi^6)$ | 17.9 |
| coarse-grained evaluation | $O(N\chi^3)$ | offline |
| pixel sampling (per pixel) | $O(\chi^3)$ | offline |

Numerically observed total runtime $\approx N\chi^{4.1}$; approximate projections at
$\sim O(\chi^3)$ are possible. Measure the shape of the curve with the demonstrator: fix
everything except $n$ and time one step.

In [ ]:
print(f"{'n':>3} {'cells':>10} {'chi':>5} {'s / step':>9}")
for ni in (5, 6, 7, 8):
    Ni = 2 ** ni
    xi = np.linspace(0, 1, Ni, endpoint=False)
    Xi, Yi = np.meshgrid(xi, xi, indexing="ij")
    psi_i = 0.05 * np.sin(2 * np.pi * Xi) * np.sin(2 * np.pi * Yi)
    fi = cfd.Flow(ni, nu=2e-3, dt=0.4 / Ni, chi_max=24, cutoff=1e-7, poisson_sweeps=1)
    ui = qq.from_grid(np.gradient(psi_i, xi, axis=1), eps=1e-9)
    vi = qq.from_grid(-np.gradient(psi_i, xi, axis=0), eps=1e-9)
    ui, vi, _ = fi.step(ui, vi)                       # warm up the pressure guess
    t0 = time.perf_counter()
    ui, vi, _ = fi.step(ui, vi)
    print(f"{ni:>3} {Ni ** 2:>10,d} {fi.rank_history[-1]:>5} {time.perf_counter() - t0:>9.2f}")

The cell count grows 64-fold across those rows; the runtime grows far less — sublinear in
cells, which is the point, though above the ideal $O(N)$ because this is unoptimised
Python and the local conjugate-gradient iterations grow with $n$. Take the *shape* of the
curve from the demonstrator and the constant from the paper.

## 3. Numerical results (Peddinti *et al.* 2024)

What the full solver — open boundaries, immersed geometry, stacked 2D encoding — does:

* **flow around a cylinder**: laminar regime, 15-bit MPS with $\chi = 30$, from
  $\mathrm{Re}=1.7$ (attached flow) through $\mathrm{Re}=25$ (steady vortex pair) to
  $\mathrm{Re}=125$ (Kármán vortex street);
* **flow around squares**: collections of squares at $\mathrm{Re}=127$ and $141$, 15-bit
  MPS with $\chi = 30$, qualitatively validated against Ansys Fluent;
* **NACA 0040 airfoil** inclined at $22^\circ$, $\mathrm{Re}=750$: 19-bit MPS with
  $\chi = 45$, $T = 5000$ steps, $28\times$ compression, with the vorticity read out by
  full resolution, coarse-grained evaluation and pixel sampling;
* further geometries in the range $100 < \mathrm{Re} < 200$.

What our demonstrator does **not** have: no ghost-cell inlet/outlet treatment (walls are
homogeneous Dirichlet), and no validation. What it does reproduce is the cost structure —
the Poisson solve dominates, rounding fights the constraint, and the rank is the thing to
watch.

## 4. Recent literature

Since the framework is flexible, it has been carried to many other problems, e.g.:

* compressible flows (Peddinti *et al.* 2025);
* incompressible Navier–Stokes with TT finite elements, chemical mixers (Kornev *et al.* 2023);
* reacting shear flows (Pinkston *et al.* 2025);
* reduced-order models for wall-bounded flows (Kiffner & Jaksch 2023);
* 2D turbulent Rayleigh–Bénard convection (van Hülst *et al.* 2026);
* fractional-step method in curvilinear coordinates (van Hülst *et al.* 2026);
* Vlasov–Maxwell and Vlasov equations (Ye & Loureiro 2024; Ameri *et al.* 2023;
  Chinomona *et al.* 2026);
* isogeometric 3D Poisson solvers and neutron transport (Tran *et al.* 2026; Ortega *et al.* 2026);
* the "magic" of fluid-dynamics data (Mello *et al.* 2025);
* and beyond physics: multi-asset option pricing (Arenstein & Kastoryano 2025).

## 5. Toward turbulence

**Yes, but** — we started from turbulence. It is in the turbulent regime that we expect a
memory advantage from scale locality, and the laminar results above do not test that.
Navier–Stokes is dissipative (not norm preserving), so the "fidelity" of a compressed
field is the wrong yardstick: we need the **statistical metrics** of turbulence. (And we
still compress with $L^2$-optimal SVD truncation; constructions tailored to the physics
are an open question.)

Pisoni *et al.* (2026) compress, evolve and synthesize homogeneous isotropic turbulence
(3D DNS, $1024^3$, $\mathrm{Re}_\lambda = 315$) in QTT, and judge it by:

* **structure functions** and Kolmogorov scale invariance,
  $S_p(r) = \langle \delta v(r)^p\rangle \sim \epsilon^{p/3} r^{p/3}$, equivalently
  $E(k) \propto \epsilon^{2/3} k^{-5/3}$;
* deviations from scale invariance = **intermittency**, seen as a non-constant
  **flatness** $\Gamma_p(r) \equiv S_p(r)/S_2(r)^{p/2}$ ($\Gamma_4 = 3$ for a Gaussian) and as
  peaked, heavy-tailed **PDFs of velocity increments** $\delta u(r)/\sigma_{\delta u(r)}$.

Their finding: interleaved ordering beats stacked, the spectrum survives strong
compression — but the small-scale statistics do not.

---
### Exercise 7 — the spectrum is the easy diagnostic

Take the synthetic **intermittent** field below (a $k^{-5/3}$ field modulated by a
lognormal multiplicative cascade). Compress it (interleaved) to $\chi \in \{16, 32, 64\}$
and for each compare against the original:

* the relative $L^2$ error,
* the energy spectrum $E(k)$,
* the flatness $\Gamma_4(r)$ for $r = 1, 2, 4, \dots, 32$ cells,
* the PDF of $\delta u(r)/\sigma_{\delta u(r)}$ at $r = 1$.

Which diagnostic recovers first as $\chi$ grows?

In [ ]:
rng = np.random.default_rng(3)
amp = K ** (-5 / 6)
amp[K > Nt / 3] = 0.0
ph = rng.standard_normal((Nt, Nt)) + 1j * rng.standard_normal((Nt, Nt))
turb = np.real(np.fft.ifft2(amp * ph))
for lvl in range(1, 7):                       # multiplicative cascade -> intermittency
    s = 2 ** lvl
    turb *= np.kron(rng.lognormal(0, 0.35, (s, s)), np.ones((Nt // s, Nt // s)))
turb /= turb.std()
seps = [1, 2, 4, 8, 16, 32]


def increments(f, r):
    return (np.roll(f, -r, axis=0) - f).ravel()


def flatness(f, r):
    d = increments(f, r)
    return (d ** 4).mean() / (d ** 2).mean() ** 2


def spectrum(f):
    E = np.abs(np.fft.fft2(f)) ** 2
    return np.bincount(np.round(K).astype(int).ravel(), E.ravel())[: Nt // 2]


# TODO

#### Solution — Exercise 7

In [ ]:
ref_E = spectrum(turb)
kk = np.arange(1, Nt // 2)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].loglog(kk, ref_E[1:], "k", lw=2, label="original")
ax[1].semilogx(seps, [flatness(turb, r) for r in seps], "k:o", lw=2, label="original")
d0 = increments(turb, 1)
hist_kw = dict(bins=81, range=(-15, 15), density=True, histtype="step")
ax[2].hist(d0 / d0.std(), color="k", lw=2, label="original", **hist_kw)

print(f"{'chi':>4} {'L2 err':>7} {'E(k) err, k<40':>15}   flatness Gamma_4(r), r = {seps}")
print(f"{'orig':>4} {'':>7} {'':>15}   {np.round([flatness(turb, r) for r in seps], 1)}")
for chi in (16, 32, 64):
    g = qq.to_grid(qq.from_grid(turb, eps=1e-14, chi_max=chi))
    E = spectrum(g)
    l2 = np.linalg.norm(g - turb) / np.linalg.norm(turb)
    e_err = np.abs(E[1:40] / ref_E[1:40] - 1).mean()
    gam = [flatness(g, r) for r in seps]
    print(f"{chi:>4} {l2:>7.3f} {100 * e_err:>14.1f}%   {np.round(gam, 1)}")
    ax[0].loglog(kk, E[1:], lw=1, label=f"$\\chi={chi}$")
    ax[1].semilogx(seps, gam, "o-", lw=1, label=f"$\\chi={chi}$")
    d = increments(g, 1)
    ax[2].hist(d / d.std(), lw=1, label=f"$\\chi={chi}$", **hist_kw)
ax[0].set(xlabel="$k$", title="energy spectrum $E(k)$")
ax[1].set(xlabel="separation $r$ (cells)", title=r"flatness $\Gamma_4(r)$")
ax[2].set(yscale="log", xlabel=r"$\delta u(1)/\sigma$", title="PDF of increments, $r=1$")
for a in ax:
    a.legend(fontsize=7)
plt.tight_layout()

The spectrum is right to a few percent long before the pointwise error is small, and the
flatness at the smallest separations **overshoots** — strong compression manufactures
spurious small-scale intermittency, the "small-scale overshooting" of the slides. $E(k)$
is the diagnostic that survives compression longest, which makes it the one least able to
tell you whether the compression was safe. Judge a compressed turbulent field on its
increment statistics.

### What is actually going on when we compress

Turbulence statistics are about the standard deviation of increments, i.e. the small
scales. Truncation smooths the field — but not uniformly: the QTT discards fine-scale
information *within* dyadic blocks, and the error concentrates at the **block
boundaries**. See it in 1D: a rough signal on $2^{12}$ points, compressed to $\chi = 8$.

In [ ]:
M1 = 12
sig = np.cumsum(np.random.default_rng(0).standard_normal(2 ** M1))
sig = (sig - sig.mean()) / sig.std()
comp = tn.qtt_to_vector(tn.qtt_from_vector(sig, eps=1e-14, chi_max=8))
err = np.abs(np.diff(comp) - np.diff(sig))          # error in the discrete derivative
pos = np.arange(1, 2 ** M1)                          # diff[i-1] straddles points i-1, i
print(f"std of d/dx: original {np.diff(sig).std():.3f}, compressed {np.diff(comp).std():.3f}")
print(f"{'block size':>10} {'derivative error on boundaries':>31} {'elsewhere':>10}")
for b in (2, 8, 64, 256):
    on = pos % b == 0
    print(f"{b:>10} {err[on].mean():>31.3f} {err[~on].mean():>10.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(sig, "k", lw=0.6, label="original")
ax[0].plot(comp, "r--", lw=0.8, label=r"QTT, $\chi=8$")
ax[0].set_title(r"$\theta(x)$"), ax[0].legend(fontsize=8)
z = slice(2040, 2060)
ax[1].plot(pos[z], np.diff(sig)[z], "k-o", ms=3, label="original")
ax[1].plot(pos[z], np.diff(comp)[z], "r--o", ms=3, label="compressed")
ax[1].axvline(2048, color="gray", lw=0.8)
ax[1].set_title(r"zoom on $\partial_x\theta$ around the block boundary $x=2048$")
ax[1].legend(fontsize=8)
plt.tight_layout()

The compressed derivative is much smoother than the original (its standard deviation
drops by more than half), and what error remains piles up at the coarse dyadic
boundaries — the larger the block, the larger the jump. That is the mechanism behind the
wrong small-scale statistics.

### The rest of the story (slides 32–38)

* **Evolution.** A 3D TT solver (initial TT snapshot → divergence-free projection →
  Euler stepping, the loop of Section 2) runs stably for 9–10 Kolmogorov time units at
  $\chi = 100$; $E(k)$ is checked against the expected spectrum over time.
* **Synthesis.** Snapshots generated *natively* in QTT: low-rank noise on the coarse
  scales plus QTT interpolation (upsampling, Guzman *et al.* 2026) for the fine ones,
  summed over the split point $m$. The result has the right spectrum and flatness, with
  $\chi$ growing linearly in the number of scales.
* **Open questions.** Truncation versus projection for the three velocity components;
  small-scale overshooting at strong compression.
* **A fix: Hybrid-TT** (passive-scalar turbulence, $4096^2$, arXiv:2608.00194). A dense
  head tensor keeps the large scales exactly, and native QTT interpolation repairs the
  block boundaries. At 1.3% of the original size it reproduces $E(k)$, $S_4(r)$, the
  flatness and the increment PDFs — better than Fourier (Galerkin) or wavelet truncation
  at the same size.

Further reading on turbulence with tensor networks: Hölscher *et al.* (2025, GPU 2D
turbulence), Gourianov *et al.* (2025, turbulence PDFs), Esmaeili *et al.* (2026, a priori
assessment for isotropic turbulence).